In [ ]:
import pandas as pd
import time
import numpy as np
import sys
sys.path.insert(0, "../../utils/")
from joblib import dump
import json
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV, KFold, train_test_split, cross_validate

In [ ]:
REPR_NAMES = [
    "embedding_antiviral_homology_90_protT5",
]

In [ ]:
MODELS = {
    "SVM": SVR,
    "KNN": KNeighborsRegressor,
    "LinearRegression": LinearRegression,
    "AdaBoost": AdaBoostRegressor,
    "XGBoost": xgb.XGBRegressor,
    "LGBM": lgb.LGBMRegressor,
    "RandomForest": RandomForestRegressor,
    "GradientBoosting": GradientBoostingRegressor,
}

In [ ]:
GRIDS = {
    "RandomForest": {
        "n_estimators": [100, 500, 1000, 3000],
        "min_samples_split": [2, 10, 20],
        "min_samples_leaf": [1, 4, 8],
        "max_features": ["sqrt", "log2"],
        "max_depth": [10, 30, None]
    },
    "AdaBoost": {
        "n_estimators": [50, 200, 500],
        "learning_rate": [0.01, 0.1, 1.0],
    },
    "GradientBoosting": {
        "n_estimators": [100, 300, 500],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 5],
        "subsample": [0.8, 1.0],
    },
    "XGBoost": {
        "n_estimators": [100, 300, 500],
        "max_depth": [3, 5],
        "learning_rate": [0.01, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "objective": ["reg:squarederror"]
    },
    "LGBM": {
        "n_estimators": [100, 300, 500],
        "max_depth": [3, 5, -1],
        "learning_rate": [0.01, 0.1],
        "num_leaves": [15, 31],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "objective": ["regression"]
    },
    "KNN": {
        "n_neighbors": [3, 5, 7],
        "weights": ["uniform", "distance"],
        "metric": ["euclidean", "manhattan"]
    },
    "SVM": {
        "C": [0.1, 1, 10],
        "kernel": ["linear", "rbf"],
        "gamma": ["scale"]
    },
    "LinearRegression": {
        "fit_intercept": [True, False]
        }
}

In [ ]:
def get_model(model_name, seed):
    model_reg = MODELS[model_name]
    if model_name == "KNN":
        model = model_reg(n_jobs=-1)
    elif model_name in ["RandomForest", "XGBoost", "LGBM"]:
        model = model_reg(random_state=seed, n_jobs=-1)
    elif model_name in ["LinearRegression", "SVM", "AdaBoost"]:
        model = model_reg()
    else:
        model = model_reg(random_state=seed)
    return model

In [ ]:
def function_split(df_data, seed):
    #Separa los datos
    train_data, val_data = train_test_split(df_data, test_size=0.2, random_state=seed)
    return train_data, val_data

In [ ]:
def metrics(model, predict_val, y_val, dataset, div):
    mae_value = mean_absolute_error(y_pred=predict_val, y_true=y_val)
    mse_value = mean_squared_error(y_pred=predict_val, y_true=y_val)
    rmse_value = np.sqrt(mse_value)
    r2_value = r2_score(y_pred=predict_val, y_true=y_val)

    df_metrics = pd.DataFrame([[dataset, model, div, mae_value, mse_value, rmse_value, r2_value]],
        columns=["dataset", "model", "sampling", "MAE", "MSE", "RMSE", "R2"]
    )

    return df_metrics

In [ ]:
def cross_function(model, X_train, y_train, cv):
    scoring_metrics = {
        "neg_mean_squared_error": "neg_mean_squared_error",
        "neg_mean_absolute_error": "neg_mean_absolute_error",
        "r2": "r2",
        "explained_variance": "explained_variance"
    }

    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring_metrics, return_train_score=True)

    df_val = pd.DataFrame({metric: scores[f'test_{metric}'] for metric in scoring_metrics})
    df_val['fit_time'] = scores['fit_time']
    df_val['Dataset'] = 'Validation'

    df_train = pd.DataFrame({metric: scores[f'train_{metric}'] for metric in scoring_metrics})
    df_train['fit_time'] = scores['fit_time']
    df_train['Dataset'] = 'Train'

    for df in [df_val, df_train]:
        df['MSE'] = -df['neg_mean_squared_error']
        df['MAE'] = -df['neg_mean_absolute_error']
        df['RMSE'] = np.sqrt(df['MSE'])
        df['R2'] = df['r2']
        df.drop(columns=['neg_mean_squared_error', 'neg_mean_absolute_error', 'r2'], inplace=True)

    results_metrics = pd.concat([df_val, df_train], ignore_index=True)

    return results_metrics

In [ ]:
def valcross_function(model, model_name, train, div, seed):
    cv = KFold(n_splits=10, shuffle=True, random_state=seed)
    results = []

    #División de los datos
    X_train = train.drop(columns="target").values
    y_train = train["target"].values

    print(f"Crossvalidation {model_name} with seed {seed} and division {div}")    

    #Valdación cruzada
    cv_scores= cross_function(model, X_train, y_train, cv).copy()
    cv_scores["model"] = model_name
    cv_scores["sampling"] = div
    results.append(cv_scores)

    all_results = pd.concat(results, ignore_index=True)
    return all_results

In [ ]:
def train_function(model, model_name, repr_name, train, val, seed, div, grid_search):
    X_train = train.drop(columns="target").values
    y_train = train["target"].values
    X_val= val.drop(columns="target").values
    y_val = val["target"].values


    #Se realiza entrenamiento del modelo
    print(f"Train {model_name} with seed {seed} and division {div}")
    start_t = time.time()
    model.fit(X_train, y_train)
    elapsed_t = time.time() - start_t
    dump(model, f"../../models_data/nobest_joblib/regression/{repr_name}_{model_name}_{seed}_{div}_reg.joblib")

    if grid_search:
        # Se realiza la búsqueda de hiperparámetros
        print(f"GridSearchCV {model_name} with seed {seed} and division {div}")
        grid = GridSearchCV(estimator=model, param_grid=GRIDS[model_name], cv=10, scoring="neg_mean_squared_error", n_jobs=-1)

        start_g = time.time()
        grid.fit(X_train, y_train)
        elapsed_g = time.time() - start_g

        # Se obtienen los mejores parámetros y el mejor modelo
        best_model = grid.best_estimator_
        dump(best_model, f"../../models_data/best_joblib/regression/{repr_name}_{model_name}_{seed}_{div}_reg_best.joblib")

        y_pred_val_grid = best_model.predict(X_val)

        val_grid_metrics = metrics(model_name, y_pred_val_grid, y_val, "grid_Validation", div)
        val_grid_metrics['fit_time'] = elapsed_g
    
        results = val_grid_metrics
    return results

In [ ]:
def main_train(df_data, seed, model_name, repr_name, valcross=False, grid_search=False):
    df_train, df_val = function_split(df_data, seed)
    model=get_model(model_name, seed)
    metrics_grid =train_function(model, model_name, repr_name, df_train, df_val, seed, "Original", grid_search)
    metrics_grid.to_csv(f"../../models_data/metrics/regression/{repr_name}_{model_name}_{seed}_metrics_reg.csv", index=False)
    if valcross:
        metrics=valcross_function(model, model_name, df_train, "Original", seed)
        metrics.to_csv(f"../../models_data/metrics/regression/{repr_name}_{model_name}_{seed}_valcross_metrics_reg.csv", index=False)

In [ ]:
seed= 42
for repr_name in REPR_NAMES:
    print(f"Loading data for representation: {repr_name}")
    df_data = pd.read_csv(f"../../data/numerical_rep_reg/{repr_name}.csv")
    if "experimental_characteristics" in df_data.columns:
        df_data.drop(["experimental_characteristics"], axis=1, inplace=True)
    for model_name in MODELS.keys():
        print(f"Processing {model_name} with {repr_name}")
        start_time = time.time() 
    
        main_train(df_data, seed, model_name, repr_name, valcross=True, grid_search=True)
    
        elapsed_time = time.time() - start_time
        print(f"Time taken for {model_name}: {elapsed_time:.2f} seconds")
        print(f"Finished {model_name}")
        print("=====================================")